# Report Figures

Generates every figure referenced in `docs/report_draft.md`, plus a few extras worth considering.
All outputs are saved to `docs/figures/` at 150 dpi (print-ready).

**Required figures (placeholders in report):**
- `score_dist.png` — reviewer score histogram  
- `aspect_freq.png` — sentence count per aspect  
- `shap_global.png` — global SHAP bar chart  
- `shap_heatmap.png` — per-hotel SHAP heatmap (top 15 hotels)  

**Extras (consider adding to report):**
- `sentiment_by_aspect.png` — positive vs negative split per aspect  
- `segment_dist.png` — reviewer segment distribution  
- `hotel_review_count.png` — hotel review count distribution (long tail)  
- `shap_city.png` — mean SHAP per aspect broken down by city  
- `review_date_dist.png` — review volume over time  

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Resolve project root
NOTEBOOK_DIR = os.path.abspath('')
ROOT = os.path.dirname(NOTEBOOK_DIR)
SRC  = os.path.join(ROOT, 'src')
for p in (ROOT, SRC):
    if p not in sys.path:
        sys.path.insert(0, p)

from paths import SENTENCES, ASPECT_SENTENCES, SHAP_SUMMARY, REVIEW_FEATURES, EVALUATION_REPORT

FIG_DIR = os.path.join(ROOT, 'docs', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

# Global style — matches Times New Roman report
plt.rcParams.update({
    'font.family':      'serif',
    'font.serif':       ['Times New Roman', 'DejaVu Serif'],
    'font.size':        9,
    'axes.titlesize':   10,
    'axes.labelsize':   9,
    'xtick.labelsize':  8,
    'ytick.labelsize':  8,
    'legend.fontsize':  8,
    'figure.dpi':       150,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

ASPECT_ORDER  = ['Room', 'Staff', 'Location', 'Food', 'Cleanliness', 'Noise']
ASPECT_COLORS = {
    'Room':        '#4C72B0',
    'Staff':       '#55A868',
    'Location':    '#C44E52',
    'Food':        '#8172B2',
    'Cleanliness': '#CCB974',
    'Noise':       '#64B5CD',
}

print('Paths OK. Figures will be saved to:', FIG_DIR)

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
print('Loading sentences.csv ...')
df_sent = pd.read_csv(SENTENCES, low_memory=False)
print(f'  {len(df_sent):,} rows')

print('Loading aspect_sentences.csv ...')
df_asp = pd.read_csv(ASPECT_SENTENCES, low_memory=False)
print(f'  {len(df_asp):,} rows')

print('Loading shap_summary.json ...')
with open(SHAP_SUMMARY) as f:
    shap_data = json.load(f)
shap_by_hotel = {e['hotel_name']: e for e in shap_data}
print(f'  {len(shap_data)} hotels (incl. __global__)')

print('Loading review_features.csv ...')
df_feat = pd.read_csv(REVIEW_FEATURES, low_memory=False)
print(f'  {len(df_feat):,} rows')

## Figure 1 — Reviewer score distribution

In [ ]:
# One score per review (df_feat has one row per review)
scores = df_feat['reviewer_score'].dropna()

fig, ax = plt.subplots(figsize=(4.5, 2.8))
ax.hist(scores, bins=np.arange(0, 11, 0.5), color='#4C72B0', edgecolor='white',
        linewidth=0.4, rwidth=0.85)
ax.axvline(scores.mean(), color='#C44E52', linewidth=1.2, linestyle='--',
           label=f'Mean = {scores.mean():.2f}')
ax.set_xlabel('Reviewer score (0–10)')
ax.set_ylabel('Number of reviews')
ax.set_title('Distribution of reviewer scores across 515,738 reviews')
ax.legend(frameon=False)
ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'score_dist.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: score_dist.png')
print(f'  Mean={scores.mean():.3f}  Median={scores.median():.1f}  SD={scores.std():.3f}')

## Figure 2 — Sentence count per aspect

In [ ]:
asp_counts = df_asp['aspect'].value_counts().reindex(ASPECT_ORDER)
colors     = [ASPECT_COLORS[a] for a in ASPECT_ORDER]

fig, ax = plt.subplots(figsize=(4.5, 2.8))
bars = ax.barh(ASPECT_ORDER[::-1], asp_counts[ASPECT_ORDER[::-1]].values,
               color=colors[::-1], height=0.6)

for bar, val in zip(bars, asp_counts[ASPECT_ORDER[::-1]].values):
    ax.text(val + 2000, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=7.5)

ax.set_xlabel('Number of labelled sentences')
ax.set_title('Sentence-level mention frequency by aspect')
ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
total = asp_counts.sum()
ax.set_xlim(0, asp_counts.max() * 1.18)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'aspect_freq.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: aspect_freq.png')
print(asp_counts.to_string())
print(f'Total matched: {total:,} / {len(df_sent):,} ({total/len(df_sent)*100:.1f}%)')

## Figure 3 — Global SHAP bar chart

In [ ]:
global_entry = shap_by_hotel['__global__']
impacts      = global_entry['aspect_impacts']

# Sort by absolute value descending
sorted_items = sorted(impacts.items(), key=lambda x: abs(x[1]), reverse=True)
labels  = [a.capitalize() for a, _ in sorted_items]
values  = [v for _, v in sorted_items]
colors  = ['#55A868' if v >= 0 else '#C44E52' for v in values]

fig, ax = plt.subplots(figsize=(4.5, 2.8))
bars = ax.barh(labels[::-1], values[::-1], color=colors[::-1], height=0.6)
ax.axvline(0, color='#333', linewidth=0.8, linestyle='--')

for bar, val in zip(bars, values[::-1]):
    offset = 0.001 if val >= 0 else -0.001
    ha     = 'left' if val >= 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:+.4f}', va='center', ha=ha, fontsize=7.5)

ax.set_xlabel('Mean SHAP value')
ax.set_title('Global aspect impact on reviewer score (XGBoost SHAP)')
pos_patch = mpatches.Patch(color='#55A868', label='Positive impact')
neg_patch = mpatches.Patch(color='#C44E52', label='Negative impact')
ax.legend(handles=[pos_patch, neg_patch], frameon=False, loc='lower right')
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'shap_global.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: shap_global.png')

## Figure 4 — Per-hotel SHAP heatmap (top 15 most-reviewed hotels)

In [ ]:
# Review counts from aspect_sentences
review_counts = df_asp.groupby('hotel_name')['review_id'].nunique().to_dict()

# Top 15 hotels by review count (excluding __global__)
top15 = sorted(
    [(h, review_counts.get(h, 0)) for h in shap_by_hotel if h != '__global__'],
    key=lambda x: x[1], reverse=True
)[:15]
top15_names = [h for h, _ in top15]

ASPECTS = ['staff', 'location', 'room', 'cleanliness', 'food', 'noise']

# Build matrix
matrix = np.array([
    [shap_by_hotel[h]['aspect_impacts'].get(a, 0) for a in ASPECTS]
    for h in top15_names
])

# Shorten hotel names
short_names = [n[:30] + ('...' if len(n) > 30 else '') for n in top15_names]

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto',
               vmin=-0.35, vmax=0.35)

ax.set_xticks(range(len(ASPECTS)))
ax.set_xticklabels([a.capitalize() for a in ASPECTS], rotation=0)
ax.set_yticks(range(len(short_names)))
ax.set_yticklabels(short_names, fontsize=7.5)
ax.set_title('Per-hotel SHAP values — 15 most-reviewed hotels', pad=10)

# Annotate cells
for i in range(len(top15_names)):
    for j in range(len(ASPECTS)):
        val = matrix[i, j]
        ax.text(j, i, f'{val:+.2f}', ha='center', va='center',
                fontsize=6.5, color='black')

cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('Mean SHAP value', fontsize=8)
cbar.ax.tick_params(labelsize=7)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'shap_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: shap_heatmap.png')

## Extra Figure A — Positive vs negative sentiment split per aspect

In [ ]:
sent_counts = (
    df_asp.groupby(['aspect', 'sentiment'])
    .size()
    .unstack(fill_value=0)
    .reindex(ASPECT_ORDER)
)
pos = sent_counts.get('Positive', pd.Series(0, index=ASPECT_ORDER))
neg = sent_counts.get('Negative', pd.Series(0, index=ASPECT_ORDER))

x   = np.arange(len(ASPECT_ORDER))
w   = 0.38
fig, ax = plt.subplots(figsize=(5.5, 3))
ax.bar(x - w/2, pos.values, w, label='Positive', color='#55A868', edgecolor='white')
ax.bar(x + w/2, neg.values, w, label='Negative', color='#C44E52', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(ASPECT_ORDER)
ax.set_ylabel('Sentence count')
ax.set_title('Positive vs negative sentence counts per aspect')
ax.legend(frameon=False)
ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'sentiment_by_aspect.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sentiment_by_aspect.png')
# Positive rate per aspect
for asp in ASPECT_ORDER:
    p = pos[asp]; n = neg[asp]
    print(f'  {asp}: {p/(p+n)*100:.1f}% positive')

## Extra Figure B — Reviewer segment distribution

In [ ]:
seg_counts = df_sent['reviewer_segment'].value_counts()
seg_order  = ['Couple', 'Solo', 'Business', 'Family', 'Group']
seg_counts = seg_counts.reindex([s for s in seg_order if s in seg_counts.index])

fig, ax = plt.subplots(figsize=(4.5, 2.8))
colors_seg = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974']
ax.barh(seg_counts.index[::-1], seg_counts.values[::-1],
        color=colors_seg[:len(seg_counts)][::-1], height=0.55)
for i, (seg, val) in enumerate(zip(seg_counts.index[::-1], seg_counts.values[::-1])):
    ax.text(val + 1000, i, f'{val:,}', va='center', fontsize=7.5)
ax.set_xlabel('Number of sentences')
ax.set_title('Sentence distribution by reviewer segment')
ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'segment_dist.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: segment_dist.png')

## Extra Figure C — Hotel review count distribution (long tail)

In [ ]:
hotel_counts = df_feat.groupby('hotel_name')['review_id'].count().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(7, 3))

# Histogram
axes[0].hist(hotel_counts, bins=40, color='#4C72B0', edgecolor='white', linewidth=0.4)
axes[0].axvline(100, color='#C44E52', linestyle='--', linewidth=1,
                label='100-review threshold')
axes[0].set_xlabel('Reviews per hotel')
axes[0].set_ylabel('Number of hotels')
axes[0].set_title('Review count distribution')
axes[0].legend(frameon=False, fontsize=7.5)

# Cumulative
sorted_counts = np.sort(hotel_counts.values)
cdf = np.arange(1, len(sorted_counts)+1) / len(sorted_counts)
axes[1].plot(sorted_counts, cdf, color='#4C72B0', linewidth=1.2)
axes[1].axvline(100, color='#C44E52', linestyle='--', linewidth=1,
                label=f'100 reviews ({(hotel_counts>=100).mean()*100:.0f}% of hotels)')
axes[1].set_xlabel('Reviews per hotel')
axes[1].set_ylabel('Cumulative fraction of hotels')
axes[1].set_title('CDF of review counts')
axes[1].legend(frameon=False, fontsize=7.5)

fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'hotel_review_count.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: hotel_review_count.png')
print(f'Hotels >= 100 reviews: {(hotel_counts>=100).sum()} ({(hotel_counts>=100).mean()*100:.1f}%)')
print(f'Hotels <  100 reviews: {(hotel_counts<100).sum()} ({(hotel_counts<100).mean()*100:.1f}%)')
print(f'Median reviews: {hotel_counts.median():.0f}')

## Extra Figure D — Review volume over time

In [ ]:
df_dated = df_sent[['review_id', 'review_date']].drop_duplicates(subset='review_id')
df_dated['review_date'] = pd.to_datetime(df_dated['review_date'], errors='coerce')
df_dated = df_dated.dropna(subset=['review_date'])
df_dated['quarter'] = df_dated['review_date'].dt.to_period('Q')

qcounts = df_dated.groupby('quarter').size()

fig, ax = plt.subplots(figsize=(5.5, 2.8))
ax.bar(range(len(qcounts)), qcounts.values, color='#4C72B0', edgecolor='white', linewidth=0.4)
ax.set_xticks(range(len(qcounts)))
ax.set_xticklabels([str(q) for q in qcounts.index], rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Reviews')
ax.set_title('Review volume by quarter (Aug 2015 – Aug 2017)')
ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'review_date_dist.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: review_date_dist.png')

## Summary — all saved figures

In [ ]:
saved = [f for f in os.listdir(FIG_DIR) if f.endswith('.png')]
print(f'{len(saved)} figures in {FIG_DIR}:')
for f in sorted(saved):
    size_kb = os.path.getsize(os.path.join(FIG_DIR, f)) // 1024
    print(f'  {f} ({size_kb} KB)')

print()
print('Required by report (replace placeholder blocks):')
required = ['score_dist.png', 'aspect_freq.png', 'shap_global.png', 'shap_heatmap.png']
for r in required:
    status = 'OK' if r in saved else 'MISSING'
    print(f'  [{status}] {r}')